# WM9B7 — PowerGraph: Cascading Failure Prediction on the IEEE-24 Power Grid

**Course:** WM9B7 — Modern Data Analytics  
**Submission:** Reproducible notebook (this file) + critical reflection report (`Critical_Reflection.pdf`) + two trimmed Python modules (`powergraph_models.py`, `powergraph_data.py`).

## What this notebook does

We predict whether an N−1 contingency on the **IEEE-24 Reliability Test System** triggers a cascading outage. The task is binary graph classification on **21,500 power-flow simulations** from the PowerGraph benchmark (Varbella et al., NeurIPS 2024); positive class rate is **20.2%**. Each graph carries 3 node features (P_net, Q_net, |V|) and 5 edge features (P_flow, Q_flow, reactance, line rating, line loading), with the IEEE-24 topology (24 buses, 38 transmission lines) shared across every simulation.

The headline architecture is **GINe** — Graph Isomorphism Network with edge-feature conditioning (Hu et al., 2020). It is trained, evaluated, and interrogated with Integrated Gradients live in this notebook; the wider experimental evidence (5-seed confidence intervals, robustness curves, conformal coverage, cross-grid transfer, ONNX deployment, SimGRACE pre-training) is loaded from a pre-computed bundle.

## How to run

1. Open in **Azure ML Studio** with kernel **`Python 3.10 - AzureML`** on a **Standard NC6s_v3 (T4 GPU)** compute target.
2. **Press *Run All*.** No local files, no manual steps. The notebook downloads a public GitHub Release tarball (~50 MB) containing the processed dataset, trained checkpoint, deterministic split indices, and pre-computed result JSONs / figures.
3. Expected end-to-end runtime: **≈ 4 minutes** on T4. The 15-minute WM9B7 cap leaves comfortable headroom even on a cold-started instance.

A CPU-only fallback works but is roughly 3× slower; the 15-minute cap should still hold.

## File map

| File | Purpose |
|---|---|
| `WM9B7_PowerGraph.ipynb` | This notebook — the reproducible pipeline. |
| `powergraph_models.py` | `GraphPooling` and `GINe_Graph` definitions. |
| `powergraph_data.py` | `PowerGraphDataset` (binary task) and split helpers. |
| `Critical_Reflection.pdf` | Critical reflection report (LO1 conceptual analysis, LO3 emerging-trends discussion). |

## Output artefacts

All results print and render **inline**. The notebook does not write to disk beyond the bundle staging directory (`./powergraph_bundle/`), which is created on first run and reused on subsequent runs.

## Reduced live-training disclosure

The live training cell runs **3 epochs** as a reproducibility demonstration, in line with WM9B7 module guidance on acceptable runtime. The marked metrics — 5-seed mean ± std, full robustness analysis, interpretability comparison — were obtained with **200 epochs and early stopping (patience = 30, typical convergence ≈ epoch 120)** over 5 random seeds (2026, 23, 456, 789, 99). Full-run metrics are loaded from `architecture_comparison_summary.json` in cell 10 and onward; the live 3-epoch numbers in cell 9 are illustrative only.

## AI-tool collaboration disclosure

This notebook was scaffolded with the assistance of **Claude (Anthropic)** under the WM9B7 *AI Collaboration* permission level. The AI was used for: code structure and documentation, debugging support, synthesis of literature citations, and drafting of expository markdown. **All numerical results, figures, and metrics were produced by the underlying Python scripts on the PowerGraph dataset; no AI tool generated metrics, model weights, or experimental outcomes.**

## Dataset attribution

PowerGraph dataset, Varbella, Briola, Aste, Cremer & Mountanios (NeurIPS 2024), distributed under **CC BY 4.0** (Figshare doi:10.6084/m9.figshare.22820534). The IEEE-24 Reliability Test System topology is in the public domain.


## 1. Environment setup

AzureML's `Python 3.10 - AzureML` kernel ships PyTorch and most scientific Python libraries, but not the PyG ecosystem (`torch-geometric`, `torch-scatter`, `torch-sparse`) or Captum. We install those against the resident PyTorch version, deriving the correct PyG wheel index from `torch.__version__` and `torch.version.cuda` so the build stays binary-compatible.


In [ ]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

import torch
TORCH_VERSION = torch.__version__.split("+")[0]
CUDA_TAG = f"cu{torch.version.cuda.replace('.', '')}" if torch.cuda.is_available() else "cpu"
PYG_INDEX = f"https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_TAG}.html"

# PyG ecosystem against the resident torch
pip_install("torch-scatter", "torch-sparse", "-f", PYG_INDEX)
pip_install("torch-geometric==2.6.1")
pip_install("captum==0.7.0", "scikit-learn>=1.3", "matplotlib", "networkx", "scipy", "pandas")

# Imports (after install)
import os, json, time, hashlib, tarfile, urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix,
)
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


## 2. Reproducibility

We seed Python, NumPy and PyTorch with **seed = 23** — the same seed used to train the bundled checkpoint (`ieee24_gine_best.pt`). cuDNN is set to deterministic mode.

**Residual non-determinism:** PyG's scatter/gather kernels do not have fully deterministic CUDA paths. Calling `torch.use_deterministic_algorithms(True)` would crash mid-training; we therefore do not enable strict mode. Live-training numbers may vary by ~0.1–0.3 percentage points across runs of the *same* seed on different GPU hardware. The marked metrics in cell 6 onwards are aggregated over 5 seeds, which absorbs this variance.


In [ ]:
import random
SEED = 23
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# torch.use_deterministic_algorithms(True)  # disabled — PyG scatter ops have non-deterministic CUDA paths
print(f"Seeded with SEED={SEED}; cuDNN deterministic={torch.backends.cudnn.deterministic}")


## 3. Fetch the precomputed artefact bundle

The bundle is a public asset on a GitHub Release. It contains:

- `ieee24_processed.pt` — the fully-processed PyG `InMemoryDataset` for the binary task.
- `ieee24_gine_best.pt` — GINe checkpoint, seed = 23, 200 epochs with early stopping.
- `ieee24_split_indices.json` — deterministic 85 / 5 / 10 train / val / test split indices.
- `results/*.json` — pre-computed result summaries from every experiment.
- `figures/*.png` — pre-rendered figures from EDA, robustness, conformal, learning curves, deployment, and SimGRACE.

A SHA-256 check guards against silent corruption of the tarball.


In [ ]:
# ---- USER-FILLED: replace these placeholders before submission ----
BUNDLE_URL = "https://github.com/<USERNAME>/wm9b7-powergraph-submission/releases/download/v1.0/submission_bundle.tar.gz"
EXPECTED_SHA256 = "<sha256-here>"
# -------------------------------------------------------------------

WORK_DIR = "./powergraph_bundle"
TARBALL = "submission_bundle.tar.gz"

os.makedirs(WORK_DIR, exist_ok=True)
tarball_path = os.path.join(WORK_DIR, TARBALL)

if not os.path.exists(tarball_path):
    print("Downloading bundle (~50 MB) ...")
    urllib.request.urlretrieve(BUNDLE_URL, tarball_path)
else:
    print("Bundle tarball already cached locally.")

with open(tarball_path, "rb") as f:
    actual_sha = hashlib.sha256(f.read()).hexdigest()
assert actual_sha == EXPECTED_SHA256, (
    f"SHA-256 mismatch — tarball may be corrupt or partially downloaded.\n"
    f"  Expected: {EXPECTED_SHA256}\n"
    f"  Actual:   {actual_sha}"
)

with tarfile.open(tarball_path) as tf:
    tf.extractall(WORK_DIR)

DATASET_PT    = os.path.join(WORK_DIR, "ieee24_processed.pt")
CHECKPOINT_PT = os.path.join(WORK_DIR, "ieee24_gine_best.pt")
SPLIT_JSON    = os.path.join(WORK_DIR, "ieee24_split_indices.json")
RESULTS_DIR   = os.path.join(WORK_DIR, "results")
FIGURES_DIR   = os.path.join(WORK_DIR, "figures")
print(f"Bundle ready. {len(os.listdir(RESULTS_DIR))} JSONs, {len(os.listdir(FIGURES_DIR))} figures.")


## 4. Brief exploratory data analysis

The PowerGraph IEEE-24 dataset contains **21,500 power-flow simulations** on the 24-bus IEEE Reliability Test System. Each graph is labelled by whether the **demand-not-served (DNS)** following an N−1 contingency exceeds zero — a **20.2% positive rate**, a severe class imbalance that motivates the class-weighted loss in cell 9. Edge features (P_flow, Q_flow, reactance, line rating, line loading) are critical to the task: the feature-ablation result in cell 12 shows that removing them collapses balanced accuracy by 24 percentage points. Topology is shared across every simulation, which makes a graph-classification framing — rather than per-edge prediction — appropriate for the binary task. Per-edge prediction was nonetheless explored as a stretch experiment; the result is loaded from `edge_level_results.json` and discussed in the report.


In [ ]:
with open(os.path.join(RESULTS_DIR, "eda_summary.json")) as f:
    eda = json.load(f)
print(json.dumps(eda, indent=2))

display(Image(os.path.join(FIGURES_DIR, "01_label_distributions.png")))
display(Image(os.path.join(FIGURES_DIR, "04_line_loading.png")))


## 5. ML-vs-DL: choice of inductive bias (LO1)

The choice between a graph neural network (GINe) and a tree ensemble (XGBoost) on this task is best framed as a choice of **inductive bias** rather than feature list. Tree ensembles encode axis-aligned splits over hand-crafted summary statistics; message-passing GNNs encode permutation equivariance and locality over the grid graph, learning end-to-end differentiable representations that respect the relational structure imposed by Kirchhoff's laws on the data-generating process. Tabular-data theory (Shwartz-Ziv & Armon, 2022; Grinsztajn et al., 2022) predicts that gradient-boosted trees should win at this scale and feature dimensionality, and the empirical results below confirm that prediction on **clean** IEEE-24 data: XGBoost reaches 0.9947 BalAcc versus GINe's 0.9890.

The interesting analytical question is therefore not whether the GNN wins on clean inputs — it does not — but **under which operational conditions** its relational inductive bias pays off. That question is answered by the robustness, cross-grid, and SimGRACE evidence loaded in cell 12, and is developed in depth in the report. This cell satisfies the WM9B7 LO1 requirement for a conceptual ML-vs-DL comparison; the longer-form analysis lives in the critical reflection.


In [ ]:
with open(os.path.join(RESULTS_DIR, "ml_baselines_summary.json")) as f:
    ml = json.load(f)
with open(os.path.join(RESULTS_DIR, "architecture_comparison_summary.json")) as f:
    arch = json.load(f)

rows = []
for name, key in [("Random Forest", "RF"), ("XGBoost", "XGBoost")]:
    r = ml["classifiers"][key]
    rows.append([
        name,
        r["test_bal_acc"]["mean"], r["test_bal_acc"]["std"],
        r["test_macro_f1"]["mean"], r["test_auroc"]["mean"], r["test_pr_auc"]["mean"],
    ])
g = arch["architectures"]["GINe"]
rows.append([
    "GINe (GNN)",
    g["test_bal_acc"]["mean"], g["test_bal_acc"]["std"],
    g["test_macro_f1"]["mean"], g["test_auroc"]["mean"], g["test_pr_auc"]["mean"],
])

df = pd.DataFrame(rows, columns=["Model", "BalAcc", "BalAcc±", "Macro-F1", "AUROC", "PR-AUC"])
print(f"5-seed test metrics (seeds = {arch['seeds']}):\n")
print(df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


## 6. Dataset and dataloaders

We load the pre-processed dataset directly via `PowerGraphDataset.from_processed`, which bypasses the raw `.mat` → PyG conversion (that step ran once at bundle-build time). The split indices come from a saved JSON, written when the checkpoint was trained, so the train / val / test partition matches exactly.


In [ ]:
from powergraph_data import PowerGraphDataset

# Load already-processed dataset directly (skip raw .mat processing)
dataset = PowerGraphDataset.from_processed(DATASET_PT, grid_name="ieee24", task="binary")
print(f"Loaded {len(dataset)} graphs.")
print(f"Sample graph: {dataset[0]}")

# Use saved deterministic split indices (matches the trained checkpoint)
with open(SPLIT_JSON) as f:
    split = json.load(f)
train_idx, val_idx, test_idx = split["train"], split["val"], split["test"]
print(f"Split: train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")

train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=16, shuffle=True)
val_loader   = DataLoader([dataset[i] for i in val_idx],   batch_size=16)
test_loader  = DataLoader([dataset[i] for i in test_idx],  batch_size=16)


## 7. Instantiate the GINe model

Hyperparameters are the Optuna-best configuration from a 100-trial search on a held-out validation seed: `hidden_dim=64`, `num_layers=4`, `dropout≈0.126`, `pooling="mean_max"`. These match the values stored in `architecture_comparison_summary.json` under `architectures.GINe.config`.


In [ ]:
from powergraph_models import GINe_Graph

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GINe_Graph(
    num_node_features=3,
    num_edge_features=5,
    hidden_dim=64,
    num_layers=4,
    num_classes=2,
    dropout=0.12602,
    pooling="mean_max",  # Optuna-best
).to(device)
print(f"Model instantiated on {device}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


## 8. Live training — 3 epochs (illustrative only)

The cell below trains GINe **live for 3 epochs** as a reproducibility demonstration, in line with WM9B7 guidance on acceptable runtime. The full project run uses **focal loss** (γ ≈ 2.02, α-cap ≈ 7.56) for **200 epochs with early stopping** (patience = 30, typical convergence ≈ epoch 120). The live cell uses **plain class-weighted cross-entropy** because it converges faster in a 3-epoch budget and produces a clearer demonstration; this is a deliberate simplification, not the marked configuration. Live numbers from this 3-epoch run are not the marked metrics — they confirm the pipeline executes. The full-run weights are loaded in cell 10.

We also record `LIVE_TRAIN_SECONDS` so that cell 13 can compute the live energy footprint from a measured wall-clock figure.


In [ ]:
# Class weights = inverse frequency, capped at 10× the minority/majority ratio
labels_train = np.array([dataset[i].y.item() for i in train_idx])
class_counts = np.bincount(labels_train, minlength=2)
class_weights = torch.tensor(
    np.minimum(len(labels_train) / (2 * class_counts), 10.0),
    dtype=torch.float32, device=device,
)
print(f"Class counts: {class_counts.tolist()}  ->  class weights: {class_weights.cpu().numpy().round(4).tolist()}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1.5305e-3, weight_decay=1.378e-6)  # Optuna-best

LIVE_EPOCHS = 3
_train_start = time.time()
for epoch in range(1, LIVE_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits = model(batch)
        loss = criterion(logits, batch.y.view(-1))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch.num_graphs

    model.eval()
    val_probs, val_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            val_probs.append(F.softmax(model(batch), dim=-1)[:, 1].cpu().numpy())
            val_labels.append(batch.y.cpu().numpy())
    val_probs = np.concatenate(val_probs); val_labels = np.concatenate(val_labels)
    val_preds = (val_probs > 0.5).astype(int)
    print(f"Epoch {epoch}/{LIVE_EPOCHS} | "
          f"train_loss={epoch_loss/len(train_idx):.4f} | "
          f"val_BalAcc={balanced_accuracy_score(val_labels, val_preds):.4f} | "
          f"val_AUROC={roc_auc_score(val_labels, val_probs):.4f}")

LIVE_TRAIN_SECONDS = time.time() - _train_start
print(f"\nLive 3-epoch training wall-clock: {LIVE_TRAIN_SECONDS:.1f} s "
      f"(used in cell 13 for energy footprint calculation).")


## 9. Load full-run checkpoint and report the marked metrics

The 3-epoch live weights now get replaced with the **200-epoch + early-stopping** weights from `ieee24_gine_best.pt` (seed = 23). Cell prints the test metrics on this single seed, the confusion matrix, and finally the **5-seed mean ± std** confidence intervals taken from `architecture_comparison_summary.json`.


In [ ]:
# Load full-run weights for the marked numbers
checkpoint_state = torch.load(CHECKPOINT_PT, map_location=device)
model.load_state_dict(checkpoint_state)
model.eval()

# Full test-set evaluation
test_probs, test_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        test_probs.append(F.softmax(model(batch), dim=-1)[:, 1].cpu().numpy())
        test_labels.append(batch.y.cpu().numpy())
test_probs = np.concatenate(test_probs); test_labels = np.concatenate(test_labels)
test_preds = (test_probs > 0.5).astype(int)

print("Test set (full-run weights, seed=23):")
print(f"  BalAcc   : {balanced_accuracy_score(test_labels, test_preds):.4f}")
print(f"  Macro-F1 : {f1_score(test_labels, test_preds, average='macro'):.4f}")
print(f"  AUROC    : {roc_auc_score(test_labels, test_probs):.4f}")
print(f"  PR-AUC   : {average_precision_score(test_labels, test_probs):.4f}")

# Confusion matrix
cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(4.0, 3.2))
im = ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color=("white" if cm[i, j] > cm.max() / 2 else "black"))
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["Safe", "Cascade"]); ax.set_yticklabels(["Safe", "Cascade"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion matrix (seed=23, full-run weights)")
plt.tight_layout(); plt.show()

# 5-seed CIs from the architecture comparison
g = arch["architectures"]["GINe"]
print(f"\n5-seed mean ± std (200 epochs each, seeds = {arch['seeds']}):")
print(f"  BalAcc   : {g['test_bal_acc']['mean']:.4f} ± {g['test_bal_acc']['std']:.4f}")
print(f"  Macro-F1 : {g['test_macro_f1']['mean']:.4f} ± {g['test_macro_f1']['std']:.4f}")
print(f"  AUROC    : {g['test_auroc']['mean']:.4f} ± {g['test_auroc']['std']:.4f}")
print(f"  PR-AUC   : {g['test_pr_auc']['mean']:.4f} ± {g['test_pr_auc']['std']:.4f}")


## 10. Interpretability — Integrated Gradients

**Integrated Gradients** (Sundararajan, Taly & Yan, 2017) attributes a model's prediction to its input features by integrating gradients along a straight-line path from a zero baseline to the input. We apply IG to a single cascade-positive test graph; the **comparison study across 200 graphs** (loaded next) shows IG outperforms PGExplainer and GNNExplainer on the human-centric BalAcc-vs-Top-K metric. The known limitation, surfaced as a critique below the figure, is IG's **negative Fidelity+** on this task: removing high-attribution edges does not reliably degrade the prediction, indicating the model relies on distributed evidence rather than a sparse subset of edges.


In [ ]:
from captum.attr import IntegratedGradients
from torch_geometric.data import Batch
import networkx as nx

# Pick one cascade-positive test graph
target_graph = next(dataset[i] for i in test_idx if dataset[i].y.item() == 1)
target_graph = target_graph.to(device)

# Wrap the model so Captum can attribute over edge_attr
def forward_for_ig(edge_attr_input):
    g = target_graph.clone()
    g.edge_attr = edge_attr_input
    return model(Batch.from_data_list([g]))

ig = IntegratedGradients(forward_for_ig)
baseline = torch.zeros_like(target_graph.edge_attr)
attributions, _ = ig.attribute(
    target_graph.edge_attr, baseline, target=1,
    return_convergence_delta=True, n_steps=30,
)
edge_importance = attributions.abs().sum(dim=1).cpu().numpy()

# Visualise on the IEEE-24 topology with NetworkX (deduplicate the bidirectional edges)
G = nx.Graph()
edge_list = target_graph.edge_index.cpu().numpy().T
seen, unique_edges, unique_imp = set(), [], []
for k, (u, v) in enumerate(edge_list):
    key = tuple(sorted((int(u), int(v))))
    if key not in seen:
        seen.add(key)
        unique_edges.append(key)
        unique_imp.append(float(edge_importance[k]))
G.add_edges_from(unique_edges)
pos = nx.kamada_kawai_layout(G)

fig, ax = plt.subplots(figsize=(8, 6))
norm_imp = np.array(unique_imp) / max(max(unique_imp), 1e-12)
nx.draw(
    G, pos, ax=ax,
    edge_color=norm_imp, edge_cmap=plt.cm.YlOrRd,
    width=2 + 5 * norm_imp,
    node_color="lightblue", node_size=400,
    with_labels=True,
)
ax.set_title("Integrated Gradients edge attributions — one cascade-positive test graph")
plt.tight_layout(); plt.show()

# Comparison study across 200 graphs
display(Markdown("**Comparison across 200 graphs** (loaded from precomputed run):"))
display(Image(os.path.join(FIGURES_DIR, "topk_sweep.png")))


**Critique.** The model-centric Fidelity+ panel on the right is the methodological caveat: IG and GNNExplainer both produce *negative* Fidelity+ for this task — removing high-attribution edges does not consistently lower the model's confidence. Rudin (2019) argues that this pattern, common in safety-critical settings, indicates post-hoc attribution is insufficient and that **inherently interpretable architectures** (e.g. prototype-based GNNs, attention-bottleneck designs) should be preferred for deployment. This is the single largest weakness in the current pipeline and the most important direction for follow-on work; it is developed at length in the report.


## 11. Pre-computed evidence gallery

The figures below summarise experiments that were too expensive to run live. Each was generated by a dedicated script in the project; the source-of-truth JSONs live alongside in `results/`.


In [ ]:
gallery = [
    ("topology_degradation.png",
     "GINe BalAcc halves under 20% random edge dropout — quantifies operational tolerance for SCADA topology gaps."),
    ("feature_noise_degradation.png",
     "At ε = 0.01 input noise, GINe retains its baseline while XGBoost drops 9.3 pp — the structural inductive bias's robustness payoff."),
    ("conformal_prediction.png",
     "Split conformal prediction matches its (1−α) coverage target; the cascade class is under-covered, motivating Mondrian conformal as future work."),
    ("learning_curve.png",
     "GINe reaches 95% of full-data performance at 20% of the training data (~3,600 graphs) — informs simulation-budget planning."),
    ("deployment_latency_comparison.png",
     "ONNX INT8 quantisation reaches 0.349 ms / graph CPU latency — 7× faster than eager GPU PyTorch and compatible with sub-second real-time contingency screening."),
    ("simgrace_tsne_comparison.png",
     "SimGRACE pre-training on multi-grid data tightens within-grid clusters, supporting cross-grid transfer (the GridFM foundation-model hypothesis)."),
]
for fname, caption in gallery:
    display(Markdown(f"**{fname.replace('.png', '').replace('_', ' ').title()}** — {caption}"))
    display(Image(os.path.join(FIGURES_DIR, fname)))


## 12. Environmental cost and limitations


In [ ]:
# Live energy footprint, computed from the wall-clock measured in cell 9.
T4_TDP_W = 70.0
GRID_INTENSITY_GCO2_PER_KWH = 126.0   # UK 2025 average (Carbon Brief, 2026)
live_wh = T4_TDP_W * (LIVE_TRAIN_SECONDS / 3600.0)
live_mg_co2 = live_wh / 1000.0 * GRID_INTENSITY_GCO2_PER_KWH * 1000.0

# Project-lifetime estimate (documented assumption):
#   ~5 architectures × 5 seeds × 200 epochs × ~3 minutes / 100 epochs on Blackwell @ 300 W TDP
#   plus robustness, conformal, MC-dropout, ablations, hyperparameter search.
#   Round number: ~25 GPU-hours @ 300 W = 7.5 kWh.
PROJECT_KWH = 7.5
project_kg_co2 = PROJECT_KWH * GRID_INTENSITY_GCO2_PER_KWH / 1000.0

display(Markdown(f'''
**Environmental cost (quantified).**
The live training run consumed approximately **{live_wh:.2f} Wh** on a T4 GPU
(70 W TDP × {LIVE_TRAIN_SECONDS:.0f} s wall-clock). At the **2025 UK grid carbon
intensity of {GRID_INTENSITY_GCO2_PER_KWH:.0f} gCO₂eq / kWh** (Carbon Brief, 2026), this
corresponds to roughly **{live_mg_co2:.0f} mgCO₂eq** — comparable to a single
LED bulb running for a few minutes. The full multi-seed × multi-architecture
experimental campaign consumed approximately **{PROJECT_KWH:.1f} kWh**
(~**{project_kg_co2*1000:.0f} gCO₂eq** at UK 2025 intensity) on RTX Blackwell hardware
across the project lifetime. This cost is amortised at deployment, where ONNX
INT8 inference at **0.349 ms / graph** consumes on the order of **0.2 Wh per
million predictions** — three orders of magnitude below the training cost when
spread across realistic operational duty cycles.

**Limitations.**
- Live training is scoped to **IEEE-24 only**; full project covers IEEE-24, IEEE-39, UK and IEEE-118 (see `cross_grid_results.json`).
- Training data is **simulated** (MATPOWER); generalisation to operational SCADA data is not validated and cannot be without partner-utility data access.
- Conformal coverage guarantees are **marginal**, not class-conditional — the cascade class is under-covered (see `conformal_summary.json`).
- Post-hoc attribution methods (IG, GNNExplainer) show **negative Fidelity+** — the model is faithful to distributed evidence rather than sparse explanations, undermining the use of these methods for case-by-case operator-facing justification.
- The 3-epoch live run is **not** the marked configuration; full-run results are seed-averaged over 5 seeds with 200 epochs of focal-loss training and early stopping. The illustrative live numbers in cell 9 should be read as a pipeline check only.
'''))


## 13. References (notebook subset, Harvard style)

The full reference list lives in the report. The following anchor references are cited directly in this notebook:

- Angelopoulos, A.N. & Bates, S. (2023) 'Conformal prediction: a gentle introduction', *Foundations and Trends in Machine Learning*, 16(4), pp. 494–591.
- Carbon Brief (2026) *Analysis: UK electricity from fossil fuels falls to record low in 2025*. Available at: https://www.carbonbrief.org (Accessed: 2 May 2026).
- Grinsztajn, L., Oyallon, E. & Varoquaux, G. (2022) 'Why do tree-based models still outperform deep learning on typical tabular data?', in *Advances in Neural Information Processing Systems 35: Datasets and Benchmarks Track*.
- Hu, W., Liu, B., Gomes, J., Zitnik, M., Liang, P., Pande, V. & Leskovec, J. (2020) 'Strategies for pre-training graph neural networks', in *Proceedings of the 8th International Conference on Learning Representations (ICLR 2020)*.
- Rudin, C. (2019) 'Stop explaining black box machine learning models for high-stakes decisions and use interpretable models instead', *Nature Machine Intelligence*, 1(5), pp. 206–215.
- Shwartz-Ziv, R. & Armon, A. (2022) 'Tabular data: deep learning is not all you need', *Information Fusion*, 81, pp. 84–90.
- Sundararajan, M., Taly, A. & Yan, Q. (2017) 'Axiomatic attribution for deep networks', in *Proceedings of the 34th International Conference on Machine Learning (ICML 2017)*, pp. 3319–3328.
- Varbella, A., Briola, A., Aste, T., Cremer, J.L. & Mountanios, F. (2024) 'PowerGraph: a power grid benchmark dataset for graph neural networks', in *Advances in Neural Information Processing Systems 37*.
- Xia, J., Wu, L., Chen, J., Hu, B. & Li, S.Z. (2022) 'SimGRACE: a simple framework for graph contrastive learning without data augmentation', in *Proceedings of the ACM Web Conference 2022 (WWW '22)*, pp. 1070–1079.

---

*End of notebook.*
